# Emodia NLP V2 — TF-IDF e BERTimbau

Este notebook usa o dataset V2 com famílias semânticas separadas por split.

Fluxo:

1. carregar o ZIP do dataset;
2. validar `train`, `validation` e `test`;
3. fazer análise exploratória;
4. treinar baseline TF-IDF + Regressão Logística;
5. avaliar no teste e no desafio imutável;
6. analisar erros por linguagem, ironia e emoção implícita;
7. treinar opcionalmente o BERTimbau;
8. comparar TF-IDF e BERTimbau;
9. exportar modelos, métricas e predições.

O split já está pronto. Não use `train_test_split`, pois isso quebraria a separação por famílias semânticas.


## 1. Instalação

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib joblib datasets transformers accelerate evaluate


## 2. Importações e configurações

In [ ]:
from pathlib import Path
import json
import random
import re
import shutil
import unicodedata
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLASSES = ["alegria", "tristeza", "ansiedade", "raiva", "medo", "nojo"]
LABEL2ID = {classe: indice for indice, classe in enumerate(CLASSES)}
ID2LABEL = {indice: classe for classe, indice in LABEL2ID.items()}

pd.set_option("display.max_colwidth", 220)

print("Ambiente configurado.")


## 3. Upload e extração do dataset

Envie o arquivo `emodia_dataset_v2_2400.zip`.


In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("Nenhum arquivo foi enviado.")

ZIP_PATH = Path(next(iter(uploaded.keys())))
PASTA_DADOS = Path("/content/emodia_dataset_v2")

if PASTA_DADOS.exists():
    shutil.rmtree(PASTA_DADOS)

PASTA_DADOS.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as arquivo_zip:
    arquivo_zip.extractall(PASTA_DADOS)

print("Arquivos extraídos:")
print(*sorted(p.name for p in PASTA_DADOS.iterdir()), sep="\n")


## 4. Carregamento dos splits

In [ ]:
def carregar_jsonl(caminho: Path) -> pd.DataFrame:
    registros = []

    with caminho.open("r", encoding="utf-8") as arquivo:
        for numero_linha, linha in enumerate(arquivo, start=1):
            linha = linha.strip()

            if not linha:
                continue

            try:
                registros.append(json.loads(linha))
            except json.JSONDecodeError as erro:
                raise ValueError(
                    f"JSON inválido em {caminho.name}, linha {numero_linha}: {erro}"
                ) from erro

    return pd.DataFrame(registros)


df_train = carregar_jsonl(PASTA_DADOS / "train.jsonl")
df_validation = carregar_jsonl(PASTA_DADOS / "validation.jsonl")
df_test = carregar_jsonl(PASTA_DADOS / "test.jsonl")
df_challenge = carregar_jsonl(PASTA_DADOS / "challenge_imutavel.jsonl")

print("Treino:", len(df_train))
print("Validação:", len(df_validation))
print("Teste:", len(df_test))
print("Desafio:", len(df_challenge))

display(df_train.head())


## 5. Validação estrutural

In [ ]:
CAMPOS_OBRIGATORIOS = {
    "id",
    "family_id",
    "fonte",
    "texto",
    "emocao_primaria",
    "split",
}

def validar_split(df: pd.DataFrame, nome: str) -> dict:
    faltantes = sorted(CAMPOS_OBRIGATORIOS - set(df.columns))

    return {
        "split": nome,
        "linhas": len(df),
        "campos_ausentes": faltantes,
        "ids_duplicados": int(df["id"].duplicated().sum()) if "id" in df else None,
        "textos_duplicados": int(
            df["texto"].str.strip().str.lower().duplicated().sum()
        ) if "texto" in df else None,
        "classes_invalidas": sorted(
            set(df["emocao_primaria"]) - set(CLASSES)
        ) if "emocao_primaria" in df else None,
    }


relatorio_splits = pd.DataFrame([
    validar_split(df_train, "train"),
    validar_split(df_validation, "validation"),
    validar_split(df_test, "test"),
])

display(relatorio_splits)


In [ ]:
familias_train = set(df_train["family_id"])
familias_validation = set(df_validation["family_id"])
familias_test = set(df_test["family_id"])

assert familias_train.isdisjoint(familias_validation)
assert familias_train.isdisjoint(familias_test)
assert familias_validation.isdisjoint(familias_test)

ids_total = pd.concat([
    df_train["id"],
    df_validation["id"],
    df_test["id"],
])

assert ids_total.duplicated().sum() == 0

print("Nenhuma família semântica aparece em mais de um split.")
print("Nenhum ID aparece em mais de um split.")


## 6. Análise exploratória

In [ ]:
def resumo_split(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    contagem = (
        df["emocao_primaria"]
        .value_counts()
        .reindex(CLASSES, fill_value=0)
    )

    return pd.DataFrame({
        "split": nome,
        "emocao": contagem.index,
        "quantidade": contagem.values,
    })


distribuicao = pd.concat([
    resumo_split(df_train, "train"),
    resumo_split(df_validation, "validation"),
    resumo_split(df_test, "test"),
], ignore_index=True)

display(distribuicao.pivot(
    index="emocao",
    columns="split",
    values="quantidade",
))


In [ ]:
for nome, dataframe in [
    ("train", df_train),
    ("validation", df_validation),
    ("test", df_test),
]:
    dataframe["numero_palavras"] = dataframe["texto"].str.split().str.len()
    dataframe["numero_caracteres"] = dataframe["texto"].str.len()

    print(f"\n=== {nome.upper()} ===")
    display(
        dataframe[
            ["numero_palavras", "numero_caracteres"]
        ].describe().T
    )


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for split in ["train", "validation", "test"]:
    dados = distribuicao[distribuicao["split"] == split]
    ax.plot(
        dados["emocao"],
        dados["quantidade"],
        marker="o",
        label=split,
    )

ax.set_title("Distribuição das emoções por split")
ax.set_xlabel("Emoção")
ax.set_ylabel("Quantidade")
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
print("Fontes no treino:")
display(df_train["fonte"].value_counts().to_frame("quantidade"))

print("\nLinguagens no treino:")
display(df_train["linguagem"].value_counts().to_frame("quantidade"))

print("\nEmoção explícita:")
display(
    pd.crosstab(
        df_train["emocao_primaria"],
        df_train["emocao_explicita"],
    )
)


## 7. Normalização textual leve

In [ ]:
def normalizar_texto(texto: str) -> str:
    if not isinstance(texto, str):
        return ""

    texto = unicodedata.normalize("NFKC", texto.lower().strip())
    texto = re.sub(r"\s+", " ", texto)

    return texto


for dataframe in [
    df_train,
    df_validation,
    df_test,
    df_challenge,
]:
    dataframe["texto_normalizado"] = (
        dataframe["texto"].apply(normalizar_texto)
    )


# Parte A — Baseline TF-IDF + Regressão Logística

In [ ]:
modelo_tfidf = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=50_000,
            sublinear_tf=True,
        ),
    ),
    (
        "classificador",
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED,
        ),
    ),
])

modelo_tfidf.fit(
    df_train["texto_normalizado"],
    df_train["emocao_primaria"],
)

print("Baseline treinado.")


In [ ]:
def avaliar_classificador(
    modelo,
    dataframe: pd.DataFrame,
    nome: str,
):
    y_real = dataframe["emocao_primaria"]
    y_pred = modelo.predict(dataframe["texto_normalizado"])

    resultado = dataframe.copy()
    resultado["emocao_prevista"] = y_pred
    resultado["acertou"] = (
        resultado["emocao_primaria"]
        == resultado["emocao_prevista"]
    )

    metricas = {
        "modelo": "tfidf_logreg",
        "conjunto": nome,
        "accuracy": accuracy_score(y_real, y_pred),
        "precision_macro": precision_score(
            y_real,
            y_pred,
            labels=CLASSES,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_real,
            y_pred,
            labels=CLASSES,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_real,
            y_pred,
            labels=CLASSES,
            average="macro",
            zero_division=0,
        ),
    }

    print(f"\n=== {nome.upper()} ===")
    display(pd.DataFrame([metricas]))

    print(
        classification_report(
            y_real,
            y_pred,
            labels=CLASSES,
            target_names=CLASSES,
            digits=4,
            zero_division=0,
        )
    )

    return resultado, metricas


resultado_tfidf_validation, metricas_tfidf_validation = avaliar_classificador(
    modelo_tfidf,
    df_validation,
    "validation",
)

resultado_tfidf_test, metricas_tfidf_test = avaliar_classificador(
    modelo_tfidf,
    df_test,
    "test",
)

resultado_tfidf_challenge, metricas_tfidf_challenge = avaliar_classificador(
    modelo_tfidf,
    df_challenge,
    "challenge",
)


In [ ]:
def plotar_matriz(
    dataframe_resultado: pd.DataFrame,
    titulo: str,
):
    matriz = confusion_matrix(
        dataframe_resultado["emocao_primaria"],
        dataframe_resultado["emocao_prevista"],
        labels=CLASSES,
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=CLASSES,
    )

    fig, ax = plt.subplots(figsize=(9, 8))
    disp.plot(ax=ax, values_format="d")
    plt.title(titulo)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


plotar_matriz(
    resultado_tfidf_test,
    "TF-IDF — matriz de confusão no teste",
)

plotar_matriz(
    resultado_tfidf_challenge,
    "TF-IDF — matriz de confusão no desafio",
)


## 8. Análise dos erros do baseline

In [ ]:
erros_tfidf = resultado_tfidf_test.loc[
    ~resultado_tfidf_test["acertou"],
    [
        "texto",
        "emocao_primaria",
        "emocao_prevista",
        "linguagem",
        "emocao_explicita",
        "contem_ironia",
        "fonte",
        "family_id",
    ],
]

print("Erros no teste:", len(erros_tfidf))
display(erros_tfidf.head(40))


In [ ]:
def desempenho_por_grupo(
    dataframe: pd.DataFrame,
    coluna: str,
) -> pd.DataFrame:
    linhas = []

    for valor, grupo in dataframe.groupby(coluna, dropna=False):
        if len(grupo) < 3:
            continue

        linhas.append({
            coluna: valor,
            "quantidade": len(grupo),
            "accuracy": grupo["acertou"].mean(),
            "f1_macro": f1_score(
                grupo["emocao_primaria"],
                grupo["emocao_prevista"],
                labels=CLASSES,
                average="macro",
                zero_division=0,
            ),
        })

    return pd.DataFrame(linhas).sort_values("accuracy")


print("Por linguagem:")
display(
    desempenho_por_grupo(
        resultado_tfidf_test,
        "linguagem",
    )
)

print("\nPor fonte:")
display(
    desempenho_por_grupo(
        resultado_tfidf_test,
        "fonte",
    )
)

print("\nPor emoção explícita:")
display(
    desempenho_por_grupo(
        resultado_tfidf_test,
        "emocao_explicita",
    )
)


# Parte B — Fine-tuning com BERTimbau

Use GPU no Colab:

`Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU`

Modelo: `neuralmind/bert-base-portuguese-cased`.


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
MODELO_BASE = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

def preparar_dataset_hf(dataframe: pd.DataFrame) -> Dataset:
    dados = dataframe[
        ["texto", "emocao_primaria"]
    ].copy()

    dados["labels"] = (
        dados["emocao_primaria"]
        .map(LABEL2ID)
        .astype(int)
    )

    dataset = Dataset.from_pandas(
        dados[["texto", "labels"]],
        preserve_index=False,
    )

    def tokenizar(lote):
        return tokenizer(
            lote["texto"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    return dataset.map(
        tokenizar,
        batched=True,
        remove_columns=["texto"],
    )


dataset_train_hf = preparar_dataset_hf(df_train)
dataset_validation_hf = preparar_dataset_hf(df_validation)
dataset_test_hf = preparar_dataset_hf(df_test)
dataset_challenge_hf = preparar_dataset_hf(df_challenge)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(dataset_train_hf)


In [ ]:
modelo_bert = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(CLASSES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)


In [ ]:
def calcular_metricas_bert(eval_pred):
    logits, labels = eval_pred
    predicoes = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predicoes),
        "precision_macro": precision_score(
            labels,
            predicoes,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            labels,
            predicoes,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            labels,
            predicoes,
            average="macro",
            zero_division=0,
        ),
    }


In [ ]:
PASTA_BERT = Path("/content/emodia_bertimbau_v2")

argumentos_treino = TrainingArguments(
    output_dir=str(PASTA_BERT),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=modelo_bert,
    args=argumentos_treino,
    train_dataset=dataset_train_hf,
    eval_dataset=dataset_validation_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=calcular_metricas_bert,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

print("Trainer configurado.")


In [ ]:
resultado_treino = trainer.train()
resultado_treino


## 9. Avaliação do BERTimbau

In [ ]:
metricas_bert_validation = trainer.evaluate(
    dataset_validation_hf,
    metric_key_prefix="validation",
)

metricas_bert_test = trainer.evaluate(
    dataset_test_hf,
    metric_key_prefix="test",
)

metricas_bert_challenge = trainer.evaluate(
    dataset_challenge_hf,
    metric_key_prefix="challenge",
)

display(pd.DataFrame([
    metricas_bert_validation,
    metricas_bert_test,
    metricas_bert_challenge,
]))


In [ ]:
def predizer_bert(
    dataset_hf,
    dataframe_original: pd.DataFrame,
) -> pd.DataFrame:
    saida = trainer.predict(dataset_hf)
    logits = saida.predictions
    ids_preditos = np.argmax(logits, axis=-1)

    probabilidades = torch.softmax(
        torch.tensor(logits),
        dim=-1,
    ).numpy()

    resultado = dataframe_original.copy()
    resultado["emocao_prevista"] = [
        ID2LABEL[int(indice)]
        for indice in ids_preditos
    ]
    resultado["confianca"] = probabilidades.max(axis=1)
    resultado["acertou"] = (
        resultado["emocao_primaria"]
        == resultado["emocao_prevista"]
    )

    return resultado


resultado_bert_test = predizer_bert(
    dataset_test_hf,
    df_test,
)

resultado_bert_challenge = predizer_bert(
    dataset_challenge_hf,
    df_challenge,
)

display(
    resultado_bert_test.loc[
        ~resultado_bert_test["acertou"],
        [
            "texto",
            "emocao_primaria",
            "emocao_prevista",
            "confianca",
            "linguagem",
            "fonte",
        ],
    ].head(40)
)


In [ ]:
plotar_matriz(
    resultado_bert_test,
    "BERTimbau — matriz de confusão no teste",
)

plotar_matriz(
    resultado_bert_challenge,
    "BERTimbau — matriz de confusão no desafio",
)


## 10. Comparação final

In [ ]:
comparacao = pd.DataFrame([
    metricas_tfidf_validation,
    metricas_tfidf_test,
    metricas_tfidf_challenge,
    {
        "modelo": "bertimbau",
        "conjunto": "validation",
        "accuracy": metricas_bert_validation["validation_accuracy"],
        "precision_macro": metricas_bert_validation["validation_precision_macro"],
        "recall_macro": metricas_bert_validation["validation_recall_macro"],
        "f1_macro": metricas_bert_validation["validation_f1_macro"],
    },
    {
        "modelo": "bertimbau",
        "conjunto": "test",
        "accuracy": metricas_bert_test["test_accuracy"],
        "precision_macro": metricas_bert_test["test_precision_macro"],
        "recall_macro": metricas_bert_test["test_recall_macro"],
        "f1_macro": metricas_bert_test["test_f1_macro"],
    },
    {
        "modelo": "bertimbau",
        "conjunto": "challenge",
        "accuracy": metricas_bert_challenge["challenge_accuracy"],
        "precision_macro": metricas_bert_challenge["challenge_precision_macro"],
        "recall_macro": metricas_bert_challenge["challenge_recall_macro"],
        "f1_macro": metricas_bert_challenge["challenge_f1_macro"],
    },
])

display(comparacao.sort_values(
    ["conjunto", "f1_macro"],
    ascending=[True, False],
))


## 11. Teste manual

In [ ]:
def prever_bertimbau(textos):
    if isinstance(textos, str):
        textos = [textos]

    entradas = tokenizer(
        textos,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    dispositivo = trainer.model.device
    entradas = {
        chave: valor.to(dispositivo)
        for chave, valor in entradas.items()
    }

    trainer.model.eval()

    with torch.no_grad():
        logits = trainer.model(**entradas).logits

    probabilidades = torch.softmax(
        logits,
        dim=-1,
    ).cpu().numpy()

    linhas = []

    for texto, probs in zip(textos, probabilidades):
        indice = int(np.argmax(probs))

        linhas.append({
            "texto": texto,
            "emocao_prevista": ID2LABEL[indice],
            "confianca": float(probs[indice]),
            "probabilidades": {
                ID2LABEL[i]: float(prob)
                for i, prob in enumerate(probs)
            },
        })

    return pd.DataFrame(linhas)


prever_bertimbau([
    "mn tô rindo aqui mas desde a briga em casa eu não consigo dormir",
    "ouvi a chave na porta e congelei",
    "pqp jogaram a culpa em mim de novo",
])


## 12. Salvamento dos artefatos

In [ ]:
PASTA_SAIDA = Path("/content/emodia_nlp_v2_resultados")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

joblib.dump(
    modelo_tfidf,
    PASTA_SAIDA / "modelo_tfidf_logreg.joblib",
)

trainer.save_model(
    PASTA_SAIDA / "modelo_bertimbau"
)

tokenizer.save_pretrained(
    PASTA_SAIDA / "modelo_bertimbau"
)

comparacao.to_csv(
    PASTA_SAIDA / "comparacao_modelos.csv",
    index=False,
)

resultado_tfidf_test.to_json(
    PASTA_SAIDA / "predicoes_tfidf_test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

resultado_bert_test.to_json(
    PASTA_SAIDA / "predicoes_bertimbau_test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

resultado_bert_challenge.to_json(
    PASTA_SAIDA / "predicoes_bertimbau_challenge.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Arquivos salvos:")
print(*sorted(p.name for p in PASTA_SAIDA.iterdir()), sep="\n")


In [ ]:
from google.colab import files

arquivo_zip = shutil.make_archive(
    "/content/emodia_nlp_v2_resultados",
    "zip",
    PASTA_SAIDA,
)

print("ZIP criado:", arquivo_zip)
files.download(arquivo_zip)


## Interpretação

Não use apenas a acurácia de validação.

Considere principalmente:

- F1 macro no teste;
- F1 macro no desafio imutável;
- desempenho nos exemplos implícitos;
- desempenho em ironia e mascaramento;
- diferença entre textos originais e novos;
- análise manual dos erros.

Mesmo que o BERTimbau supere o TF-IDF, o dataset continua sintético. O modelo ainda precisa ser validado em relatos reais, anonimizados e coletados com consentimento.
